# 06 — AlphaEarth 128x128 patches (EE export + local patch cut)

EE has no streaming-read equivalent of `odc_load`, so each tile is exported as a GeoTIFF via `ee.batch` (2023, 2024 — annual, 2 steps not ~150), then patches are cut locally, same as S1/S2.

Encoding: int16, `scale_factor=1/32767` (embedding documented range [-1,1], verified below), nodata sentinel -32768.

Output per tile: `tile_<tx>_<ty>.zarr.zip` `(point, year, band, 128, 128)` int16, `.meta.json`, `.points.parquet` — same layout as S1/S2.

In [1]:
!pip -q install earthengine-api geopandas rioxarray "zarr<3" pyarrow

In [2]:
import os, json, glob, time, shutil, warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import rioxarray
import zarr
assert zarr.__version__.startswith('2.'), (
    f"zarr {zarr.__version__} installed -- this notebook needs zarr 2.x "
    "(zarr.Blosc / zarr.open(..., compressor=...) are v2 APIs, removed in v3). "
    "Restart the runtime after the pip cell above so the pin actually takes effect: "
    "Runtime -> Restart session, then re-run from the top WITHOUT re-running pip "
    "(or re-run pip then restart again).")
print("zarr version OK:", zarr.__version__)
import ee

warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

ROOT        = '/content/drive/MyDrive/Crop_Classification'
DIR_POINTS  = f'{ROOT}/01_Points'
DIR_PATCHES = f'{ROOT}/04_Patches'
DIR_QA      = f'{ROOT}/05_QA'
os.makedirs(DIR_PATCHES, exist_ok=True); os.makedirs(DIR_QA, exist_ok=True)

# Local scratch on the Colab VM. Never point this at Drive.
WORK_DIR = '/content/work_ae'
os.makedirs(WORK_DIR, exist_ok=True)

POINTS_GPKG  = f'{DIR_POINTS}/wbcrop_points_extended.gpkg'
POINTS_LAYER = 'wbcrop_points_extended'

EE_PROJECT = 'ee-geographymanas'
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

free_gb = shutil.disk_usage('/content').free / 1e9
print("points :", POINTS_GPKG, "| scratch free:", f"{free_gb:,.0f} GB")

zarr version OK: 2.18.7
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
points : /content/drive/MyDrive/Crop_Classification/01_Points/wbcrop_points_extended.gpkg | scratch free: 93 GB


## Configuration

In [3]:
TEST_MODE      = True     # True = one tile, few points, full audit. False = full state.
TEST_N_POINTS  = 15
TEST_TILE_RANK = 0

YEARS = [2023, 2024]      # every calendar year AlphaEarth can offer for your window
EXPORT_FOLDER  = 'CropClass_AlphaEarth_Patches'   # Drive folder EE exports land in

PATCH    = 128
RES_M    = 10
TILE_DEG = 0.1
HALO_DEG = 0.008          # > half a patch (640 m) so edge points get a full window

NODATA       = -32768     # reserved sentinel, outside the valid encoded range
SCALE_FACTOR = 1 / 32767
MAX_RETRIES  = 3
FORCE        = False

RUN_ID = f"alphaearth_{PATCH}px_{'_'.join(map(str, YEARS))}"
if TEST_MODE:
    RUN_ID += "_TEST"
STORE_DIR = os.path.join(DIR_PATCHES, RUN_ID)
os.makedirs(STORE_DIR, exist_ok=True)

col = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
print("MODE  :", "TEST" if TEST_MODE else "FULL")
print("YEARS :", YEARS, "| STORE:", STORE_DIR)

MODE  : TEST
YEARS : [2023, 2024] | STORE: /content/drive/MyDrive/Crop_Classification/04_Patches/alphaearth_128px_2023_2024_TEST


## Verify the embedding range

Measured from a real sample, not assumed.

In [4]:
pts = gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER).to_crs('EPSG:4326')
assert pts['id'].is_unique, "duplicate ids in the points file"

probe_pt = ee.Geometry.Point([float(pts.geometry.x.iloc[0]), float(pts.geometry.y.iloc[0])])
probe_img = col.filterDate('2024-01-01', '2025-01-01').filterBounds(probe_pt).first()
vals = np.array(list(probe_img.reduceRegion(
    ee.Reducer.toList(), probe_pt.buffer(15), scale=10).getInfo().values())[0] or [])
print("sample pixel, all 64 bands:")
print(f"  min {vals.min():.4f} | max {vals.max():.4f} | "
      f"L2 norm {np.linalg.norm(vals):.4f}")
assert vals.min() >= -1.01 and vals.max() <= 1.01, (
    "values fall outside [-1,1] -- STOP and recompute SCALE_FACTOR before extracting.")
print("\nWithin [-1, 1] as documented. int16 encoding at scale 1/32767 confirmed safe.")

sample pixel, all 64 bands:
  min 0.0039 | max 0.0325 | L2 norm 0.0462

Within [-1, 1] as documented. int16 encoding at scale 1/32767 confirmed safe.


## Points and tiling

In [5]:
pts['tx'] = np.floor(pts.geometry.x / TILE_DEG).astype(int)
pts['ty'] = np.floor(pts.geometry.y / TILE_DEG).astype(int)
tiles_all = pts.groupby(['tx','ty']).size().sort_values(ascending=False)

if TEST_MODE:
    tkey = tiles_all.index[TEST_TILE_RANK]
    pts = pts[(pts['tx']==tkey[0]) & (pts['ty']==tkey[1])].head(TEST_N_POINTS).copy()
    print(f"TEST: tile {tkey}, {len(pts)} points, crops: {sorted(pts['crop'].unique())}")

tiles = pts.groupby(['tx','ty']).size().sort_values(ascending=False)
N_INPUT = len(pts)
print(f"Points: {N_INPUT:,} | tiles: {len(tiles)}")

def tile_bbox(tx, ty, buf=0.0):
    return [tx*TILE_DEG - buf, ty*TILE_DEG - buf,
            (tx+1)*TILE_DEG + buf, (ty+1)*TILE_DEG + buf]

def utm_epsg(lon, lat):
    zone = int((lon + 180) // 6) + 1
    return f"EPSG:{(32600 if lat >= 0 else 32700) + zone}"

tile_crs = {(tx,ty): utm_epsg((tile_bbox(tx,ty)[0]+tile_bbox(tx,ty)[2])/2,
                              (tile_bbox(tx,ty)[1]+tile_bbox(tx,ty)[3])/2)
            for (tx,ty) in tiles.index}

TEST: tile (np.int64(882), np.int64(265)), 15 points, crops: ['pine_apple']
Points: 15 | tiles: 1


## Submit exports

One export per (tile, year); masked pixels unmasked to NODATA before export.

In [6]:
def task_name(tx, ty, year):
    return f"AEpatch_{year}_{tx}_{ty}"

tasks = []
for (tx, ty) in tiles.index:
    crs  = tile_crs[(tx, ty)]
    geom = ee.Geometry.Rectangle(tile_bbox(tx, ty, HALO_DEG), proj='EPSG:4326', geodesic=False)
    for year in YEARS:
        img = (col.filterDate(f'{year}-01-01', f'{year+1}-01-01').mosaic()
               .multiply(32767).round().toInt16()
               .unmask(NODATA))
        name = task_name(tx, ty, year)
        t = ee.batch.Export.image.toDrive(
            image=img, description=name, folder=EXPORT_FOLDER,
            region=geom, scale=RES_M, crs=crs, maxPixels=1e9, fileFormat='GeoTIFF')
        t.start()
        tasks.append((name, tx, ty, year, t))

print(f"{len(tasks)} export tasks submitted ({len(tiles)} tiles x {len(YEARS)} years).")
print("Monitor: https://code.earthengine.google.com/tasks")

2 export tasks submitted (1 tiles x 2 years).
Monitor: https://code.earthengine.google.com/tasks


## Wait for exports

In [8]:
def poll(tasks, interval=20, timeout_min=180):
    t0 = time.time()
    while True:
        states = [t.status()['state'] for *_, t in tasks]
        from collections import Counter
        c = Counter(states)
        el = (time.time() - t0) / 60
        print(f"[{el:.1f} min] {dict(c)}")
        if c.get('COMPLETED', 0) + c.get('FAILED', 0) + c.get('CANCELLED', 0) == len(tasks):
            break
        if el > timeout_min:
            print("Timeout -- re-run this cell to keep waiting."); break
        time.sleep(interval)
    bad = [(n, tx, ty, y) for n, tx, ty, y, t in tasks if t.status()['state'] != 'COMPLETED']
    if bad:
        print(f"\n{len(bad)} tasks did not complete:", bad[:10])
    return bad

failed_exports = poll(tasks)

[0.0 min] {'COMPLETED': 2}


## Cut patches from the exported tiles

One tile at a time: copy its 1-2 GeoTIFFs to local scratch, read with rioxarray, find each
point's pixel window, stack years along the time axis, write zarr, zip, move to `STORE_DIR`.

In [10]:
HALF = PATCH // 2
EXPORT_DIR = f'/content/drive/MyDrive/{EXPORT_FOLDER}'

def process_tile(tx, ty, tp):
    zip_path = os.path.join(STORE_DIR, f"tile_{tx}_{ty}.zarr.zip")
    meta_p   = os.path.join(STORE_DIR, f"tile_{tx}_{ty}.meta.json")
    if os.path.exists(meta_p) and os.path.exists(zip_path) and not FORCE:
        return 'skipped'

    crs = tile_crs[(tx, ty)]
    arrays = []
    for year in YEARS:
        src = os.path.join(EXPORT_DIR, f"{task_name(tx, ty, year)}.tif")
        if not os.path.exists(src):
            return 'no_export'
        local_tif = os.path.join(WORK_DIR, os.path.basename(src))
        shutil.copy(src, local_tif)
        da = rioxarray.open_rasterio(local_tif)
        arrays.append(da)
        os.remove(local_tif)

    n_b = arrays[0].shape[0]
    n_t = len(YEARS)
    n_p = len(tp)
    xs = arrays[0]['x'].values
    ys = arrays[0]['y'].values

    tpp = tp.to_crs(crs)
    px = np.array([int(np.argmin(np.abs(xs - v))) for v in tpp.geometry.x.values])
    py = np.array([int(np.argmin(np.abs(ys - v))) for v in tpp.geometry.y.values])
    px = np.clip(px, HALF, len(xs) - HALF)
    py = np.clip(py, HALF, len(ys) - HALF)

    local = os.path.join(WORK_DIR, f"tile_{tx}_{ty}.zarr")
    if os.path.exists(local):
        shutil.rmtree(local)
    comp = zarr.Blosc(cname='zstd', clevel=5, shuffle=zarr.Blosc.BITSHUFFLE)
    z = zarr.open(local, mode='w', shape=(n_p, n_t, n_b, PATCH, PATCH),
                  chunks=(1, 1, n_b, PATCH, PATCH), dtype='int16', compressor=comp)

    for ti, da in enumerate(arrays):
        stack = da.values
        for pi in range(n_p):
            y0, x0 = py[pi] - HALF, px[pi] - HALF
            z[pi, ti] = stack[:, y0:y0+PATCH, x0:x0+PATCH]
        del stack
    for da in arrays:
        da.close()

    ids = [int(v) for v in tp['id'].values]
    z.attrs.update({
        'run_id': RUN_ID, 'source': 'alphaearth', 'collection':
            'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL',
        'dims': ['point','year','band','y','x'], 'bands': [f'A{i:02d}' for i in range(n_b)],
        'dtype': 'int16', 'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
        'encoding': 'unit_embedding_x32767',
        'crs': crs, 'resolution_m': RES_M, 'patch': PATCH,
        'years': YEARS, 'point_ids': ids,
        'decode': 'value * scale_factor where value != nodata',
    })
    zarr.consolidate_metadata(local)
    tmp_zip = os.path.join(WORK_DIR, f"tile_{tx}_{ty}.zarr")
    shutil.make_archive(tmp_zip, 'zip', local)
    shutil.move(tmp_zip + '.zip', zip_path)
    shutil.rmtree(local)

    res = RES_M
    samples = []
    for pi, pid in enumerate(ids):
        x_ul = float(xs[px[pi]-HALF]); y_ul = float(ys[py[pi]-HALF])
        samples.append({
            'id': pid, 'array_index': pi,
            'transform': [res, 0.0, x_ul, 0.0, -res, y_ul],
            'bounds_utm': [x_ul, y_ul - res*PATCH, x_ul + res*PATCH, y_ul],
            'center_lonlat': [float(tp.geometry.x.values[pi]), float(tp.geometry.y.values[pi])],
            'crop': str(tp['crop'].values[pi]), 'district': str(tp['district'].values[pi]),
            'collection_date': str(tp['collection_date'].values[pi])
                               if 'collection_date' in tp else None,
        })
    tile_meta = {
        'run_id': RUN_ID, 'source': 'alphaearth', 'tile': [int(tx), int(ty)],
        'store': os.path.basename(zip_path),
        'shape': [n_p, n_t, n_b, PATCH, PATCH],
        'dims': ['point','year','band','y','x'],
        'bands': [f'A{i:02d}' for i in range(n_b)], 'dtype': 'int16',
        'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
        'encoding': 'unit_embedding_x32767',
        'crs': crs, 'resolution_m': res, 'patch': PATCH,
        'n_points': n_p, 'years': YEARS, 'point_ids': ids, 'samples': samples,
    }
    with open(meta_p, 'w') as f:
        json.dump(tile_meta, f, indent=1)

    pd.DataFrame({'id': ids, 'array_index': np.arange(n_p),
                  'store': os.path.basename(zip_path),
                  'lon': tp.geometry.x.values, 'lat': tp.geometry.y.values,
                  'crop': tp['crop'].values, 'district': tp['district'].values,
                  'crs': crs, 'tile_tx': tx, 'tile_ty': ty}
                 ).to_parquet(os.path.join(STORE_DIR, f"tile_{tx}_{ty}.points.parquet"),
                               index=False)
    return 'done'


def process_retry(tx, ty, tp):
    for a in range(1, MAX_RETRIES + 1):
        try:
            return process_tile(tx, ty, tp)
        except Exception as e:
            if a == MAX_RETRIES:
                print(f"  tile {tx}_{ty} FAILED: {type(e).__name__}: {str(e)[:90]}")
                return 'error'
            time.sleep(2 ** a)
    return 'error'

## Run

In [11]:
def dirsize(p):
    return sum(os.path.getsize(os.path.join(d, f))
               for d, _, fs in os.walk(p) for f in fs)

if TEST_MODE:
    (tx0, ty0) = tiles.index[0]
    t0 = time.time()
    r  = process_retry(tx0, ty0, pts)
    dt = time.time() - t0
    zip0 = os.path.join(STORE_DIR, f"tile_{tx0}_{ty0}.zarr.zip")
    print("result:", r, f"| {dt:.0f}s")
    if r == 'done':
        mb = os.path.getsize(zip0) / 1e6
        m  = json.load(open(os.path.join(STORE_DIR, f"tile_{tx0}_{ty0}.meta.json")))
        print(f"{m['n_points']} points x {len(m['years'])} years -> {mb:,.1f} MB")
        full_pts = len(gpd.read_file(POINTS_GPKG, layer=POINTS_LAYER))
        proj_gb = mb / m['n_points'] * full_pts / 1e3
        print(f"\nPROJECTED full run ({len(tiles_all)} tiles, {full_pts:,} points): "
              f"~{proj_gb:,.1f} GB")
        print("Far smaller than S1/S2 -- this is 1-2 annual steps, not a full series.")
else:
    failed = []
    stats = {'done':0,'skipped':0,'no_export':0,'error':0}
    t0 = time.time()
    for i, ((tx, ty), cnt) in enumerate(tiles.items(), 1):
        r = process_retry(tx, ty, pts[(pts['tx']==tx)&(pts['ty']==ty)])
        stats[r] += 1
        if r in ('error', 'no_export'):
            failed.append({'tx': int(tx), 'ty': int(ty), 'reason': r})
        if i % 20 == 0 or i == len(tiles):
            el = time.time() - t0
            print(f"[{i}/{len(tiles)}] {stats} | {el/60:.1f} min")

result: skipped | 0s


## TIMESTEP AUDIT

Confirms both years are present, per point, and decodes back to plausible embedding values.

In [12]:
stores = sorted(glob.glob(os.path.join(STORE_DIR, "tile_*.zarr.zip")))
assert stores, "no store written -- check the run cell"
s0 = stores[0]
zs = zarr.ZipStore(s0, mode='r')
z  = zarr.open(zs, mode='r')
A  = z.attrs.asdict()
n_p, n_t, n_b = z.shape[0], z.shape[1], z.shape[2]

print("array shape:", z.shape, "| years:", A['years'], "| dtype:", z.dtype)
assert n_t == len(YEARS), f"expected {len(YEARS)} year-steps, array has {n_t}"

valid = np.zeros((n_p, n_t), dtype='float32')
for pi in range(n_p):
    for ti in range(n_t):
        valid[pi, ti] = float((z[pi, ti] != A['nodata']).mean())
print(f"\nvalid-pixel fraction: min {valid.min():.2f} | median {np.median(valid):.2f}")
print(f"entirely-nodata (point,year) cells: {int((valid==0).sum())} / {valid.size}")

pi = 0
for ti, yr in enumerate(A['years']):
    patch = z[pi, ti]
    v = patch[patch != A['nodata']]
    dec = v.astype('float32') * A['scale_factor']
    print(f"year {yr}: valid {float((patch!=A['nodata']).mean()):.1%} | "
          f"decoded range {dec.min():.4f} .. {dec.max():.4f}")
print("\nExpected: roughly within [-1, 1], matching the probe cell above.")

mf = sorted(glob.glob(os.path.join(STORE_DIR, "tile_*.meta.json")))
M = json.load(open(s0.replace(".zarr.zip", ".meta.json")))
ok = (M['years'] == YEARS and len(M['samples']) == n_p
      and [s['array_index'] for s in M['samples']] == list(range(n_p)))
print("\nJSON agrees with the array:", ok)
assert ok, "tile JSON does not match the array"

array shape: (15, 2, 64, 128, 128) | years: [2023, 2024] | dtype: int16

valid-pixel fraction: min 1.00 | median 1.00
entirely-nodata (point,year) cells: 0 / 30
year 2023: valid 100.0% | decoded range -0.4136 .. 0.4656
year 2024: valid 100.0% | decoded range -0.4036 .. 0.4656

Expected: roughly within [-1, 1], matching the probe cell above.

JSON agrees with the array: True


In [13]:
qa = {
    'generated': pd.Timestamp.now().isoformat(),
    'mode': 'TEST' if TEST_MODE else 'FULL',
    'run_id': RUN_ID, 'source': 'alphaearth', 'years': YEARS,
    'patch': PATCH, 'dtype': 'int16', 'scale_factor': SCALE_FACTOR, 'nodata': NODATA,
    'points': int(n_p), 'valid_fraction_median': float(np.median(valid)),
}
with open(os.path.join(DIR_QA, f'06_patches_{RUN_ID}_qa.json'), 'w') as f:
    json.dump(qa, f, indent=2, default=str)
print("QA saved.")

QA saved.


---
### Before `TEST_MODE = False`
- Probe must confirm values within [-1,1].
- `n_t == len(YEARS)` for every tile.
- Size projection should be tens of GB, not TB.

### Next: Tessera patch notebook